In [ ]:
!pip install kaggle -q
from google.colab import files
files.upload()   # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plantvillage_raw
import os
print(os.listdir("plantvillage_raw"))

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:28<00:00, 77.3MB/s]

['plantvillage dataset']


In [ ]:
import os, random, shutil
import numpy as np

RNG = 42
random.seed(RNG); np.random.seed(RNG)

RAW_DIR = "plantvillage_raw/plantvillage dataset/color"
SPLIT_DIR = "data/images"
CROP_PREFIXES = ["Potato", "Corn_(maize)", "Soybean"]
MAX_PER_CLASS = 300

def make_stratified_split(raw_dir=RAW_DIR, split_dir=SPLIT_DIR,
                           crop_prefixes=CROP_PREFIXES, max_per_class=MAX_PER_CLASS, val_frac=0.2):
    class_dirs = [d for d in sorted(os.listdir(raw_dir)) if any(d.startswith(p) for p in crop_prefixes)]
    counts = {}
    for cls in class_dirs:
        src = os.path.join(raw_dir, cls)
        f_list = [f for f in os.listdir(src) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        random.shuffle(f_list); f_list = f_list[:max_per_class]
        n_val = max(1, int(len(f_list) * val_frac))
        val_f, train_f = f_list[:n_val], f_list[n_val:]
        for split_name, split_files in [("train", train_f), ("val", val_f)]:
            dst = os.path.join(split_dir, split_name, cls)
            os.makedirs(dst, exist_ok=True)
            for f in split_files:
                shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        counts[cls] = {"train": len(train_f), "val": len(val_f)}
    return counts

counts = make_stratified_split(raw_dir=RAW_DIR)
for cls, c in counts.items():
    print(f"  {cls:<45} train={c['train']:4d}  val={c['val']:4d}")

  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot train= 240  val=  60
  Corn_(maize)___Common_rust_                   train= 240  val=  60
  Corn_(maize)___Northern_Leaf_Blight           train= 240  val=  60
  Corn_(maize)___healthy                        train= 240  val=  60
  Potato___Early_blight                         train= 240  val=  60
  Potato___Late_blight                          train= 240  val=  60
  Potato___healthy                              train= 122  val=  30
  Soybean___healthy                             train= 240  val=  60


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/agrivision', exist_ok=True)
!cp -r data /content/drive/MyDrive/agrivision/
print("Backup contents:", os.listdir('/content/drive/MyDrive/agrivision'))

Mounted at /content/drive
Backup contents: ['data', 'features']


# 04 — Image Preprocessing (Feature Extraction)
### AgriVision AI — Disease Detection branch

## Classical-CV feature extraction (HSV histogram + LBP texture + pixel ratios)

In [ ]:
import cv2
import numpy as np
from skimage.feature import local_binary_pattern

IMG_SIZE = 128

def hsv_histogram(img_bgr, bins=16):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hist = []
    for ch in range(3):
        h = cv2.calcHist([hsv], [ch], None, [bins], [0, 256])
        hist.append(cv2.normalize(h, h).flatten())
    return np.concatenate(hist)

def lbp_histogram(img_bgr, n_points=24, radius=3):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, n_points, radius, method="uniform")
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins), density=True)
    return hist

def dark_and_brown_ratio(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2]
    dark_ratio = float(np.mean(v < 60))
    h, s = hsv[:, :, 0], hsv[:, :, 1]
    brown_mask = (h >= 5) & (h <= 25) & (s > 60) & (v > 20) & (v < 200)
    return np.array([dark_ratio, float(np.mean(brown_mask))])

def extract_features(img_bgr):
    img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    return np.concatenate([
        hsv_histogram(img_bgr), lbp_histogram(img_bgr), dark_and_brown_ratio(img_bgr)
    ]).astype(np.float32)

FEATURE_NAMES = (
    [f"hsv_h_{i}" for i in range(16)] + [f"hsv_s_{i}" for i in range(16)] +
    [f"hsv_v_{i}" for i in range(16)] + [f"lbp_{i}" for i in range(26)] +
    ["dark_pixel_ratio", "brownish_pixel_ratio"]
)
print(f"Feature vector length: {len(FEATURE_NAMES)}")

Feature vector length: 76


## Build train/val feature matrices

In [ ]:
def build_dataset(split_dir=SPLIT_DIR, split="train"):
    root = os.path.join(split_dir, split)
    classes = sorted(os.listdir(root))
    X, y, paths = [], [], []
    for ci, cls in enumerate(classes):
        cls_dir = os.path.join(root, cls)
        for f in os.listdir(cls_dir):
            fp = os.path.join(cls_dir, f)
            img = cv2.imread(fp)
            if img is None:
                continue
            X.append(extract_features(img)); y.append(ci); paths.append(fp)
    X = np.array(X, dtype=np.float32); y = np.array(y, dtype=np.int64)
    print(f"[{split}] {len(X)} images | {X.shape[1]} features | {len(classes)} classes")
    return X, y, classes, paths

Xtr, ytr, classes, _ = build_dataset(split="train")
Xva, yva, classes_va, val_paths = build_dataset(split="val")
assert classes == classes_va, "train/val class order mismatch"
n_classes = len(classes)

[train] 1802 images | 76 features | 8 classes
[val] 450 images | 76 features | 8 classes


## CNN-ready image datasets (for MobileNetV2 / EfficientNetB0 in notebook 05)

In [ ]:
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))
# Set Runtime -> Change runtime type -> T4 GPU before running this cell for real speed.

IMG_SIZE_CNN = 224
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_DIR}/train", image_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
    batch_size=BATCH_SIZE, label_mode="int", shuffle=True, seed=42)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_DIR}/val", image_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
    batch_size=BATCH_SIZE, label_mode="int", shuffle=False)

cnn_classes = train_ds.class_names
assert cnn_classes == classes, "class order mismatch between classical and CNN pipelines"

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
print("CNN datasets ready:", cnn_classes)


GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 1802 files belonging to 8 classes.
Found 450 files belonging to 8 classes.
CNN datasets ready: ['Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Soybean___healthy']


## Save feature arrays to Drive so 05 doesn't need to re-extract them

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/agrivision/features', exist_ok=True)
np.save('/content/drive/MyDrive/agrivision/features/Xtr.npy', Xtr)
np.save('/content/drive/MyDrive/agrivision/features/ytr.npy', ytr)
np.save('/content/drive/MyDrive/agrivision/features/Xva.npy', Xva)
np.save('/content/drive/MyDrive/agrivision/features/yva.npy', yva)
import json
with open('/content/drive/MyDrive/agrivision/features/classes.json', 'w') as f:
    json.dump({"classes": classes, "val_paths": val_paths}, f)
print("Saved:", os.listdir('/content/drive/MyDrive/agrivision/features'))

Saved: ['Xva.npy', 'yva.npy', 'ytr.npy', 'classes.json', 'Xtr.npy']
